# Intake human-marathon dataset quality audit

## tl;dr

This 10,000-row corpus is a deterministic template/tone parser stress fixture, not observed seeker demand. It contains only 22 unique normalized texts (9,978 duplicate rows; 99.78%), despite 10,000 records. All rows passed the fixture's parser checks, but only 6 of 18 real-estate leaves, 9 cities and 16 neighborhoods occur. The pass result is closed-loop and does not estimate real-user accuracy. It is not a Laya typed-decision training dataset.


## Context & Methods

**Grain:** one generated Persian need-style input paired with one parser result.

**Intended use:** determine whether this existing fixture is suitable evidence for the real-estate Laya fine-tuning objective.

**Sources:** `data/intake-human-marathon/run-2026-06-19T16-22-45/state.json`, `next-needs.jsonl`, and `results.jsonl`. The generator source is `scripts/stress/intake-marathon/human-need-composer.ts`; the run state records `useLlmCompose=false`.

The audit loads the records but never displays or writes the Persian text. It checks row counts, source/tone/category mix, expected-to-resolved category agreement, city/neighborhood resolution counts, parser pass status, duplicated normalized text, real-estate category coverage, and whether consent/provenance/review/typed-target fields exist. No chart is used because the evidence is a bounded coverage and validity audit rather than a trend or distribution decision.

### Key Assumptions

The real-estate leaf list below is the app's current taxonomy as mapped in `docs/divar-laya-derived-corpus-audit-2026-09-25.md`. A result marked `ok` means the fixture evaluator passed; it does not mean a real user confirmed the label.

## Data

Load only the local snapshot used for this audit. All printed outputs are aggregates; raw text and row identifiers are deliberately excluded.

In [ ]:
from collections import Counter
from pathlib import Path
import json
import re
import unicodedata

project_root = Path.cwd()
while not (project_root / 'data/intake-human-marathon').is_dir() and project_root.parent != project_root:
    project_root = project_root.parent
assert (project_root / 'data/intake-human-marathon').is_dir(), 'Run this notebook from the repository or a descendant directory.'
run_dir = project_root / 'data/intake-human-marathon/run-2026-06-19T16-22-45'

def read_jsonl(path):
    with path.open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]

state = json.loads((run_dir / 'state.json').read_text(encoding='utf-8'))
needs = read_jsonl(run_dir / 'next-needs.jsonl')
results = read_jsonl(run_dir / 'results.jsonl')
len(needs), len(results), state['total'], state['completed'], state['useLlmCompose']

## Results

### Profile and structural checks

The following cell reports aggregate counts and asserts the stable row-level invariants.

In [ ]:
estate_leaves = {
    'apartment-sale', 'apartment-rent', 'villa-sale', 'villa-rent',
    'land-sale', 'land-rent', 'shop-sale', 'shop-rent',
    'office-sale', 'office-rent', 'industrial-sale', 'industrial-rent',
    'suite-apartment-rent', 'villa-short-rent', 'workspace-short-rent',
    'pre-sale-services', 'construction-partnership', 'agency-services',
}
source_counts = Counter(row.get('source') for row in results)
tone_counts = Counter(row.get('tone') for row in needs)
category_counts = Counter(row.get('category') for row in needs)
result_category_counts = Counter(row.get('categorySlug') for row in results)
resolved_cities = {row.get('resolvedCity') for row in results if row.get('resolvedCity')}
resolved_neighborhoods = {row.get('resolvedNeighborhood') for row in results if row.get('resolvedNeighborhood')}

def normalize_text(text):
    normalized = unicodedata.normalize('NFKC', text).casefold()
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

normalized_texts = [normalize_text(row['text']) for row in needs]
duplicate_text_rows = len(normalized_texts) - len(set(normalized_texts))
matched_expected_category = sum(
    need.get('category') == result.get('categorySlug')
    for need, result in zip(needs, results, strict=True)
)
estate_categories = set(category_counts) & estate_leaves
missing_estate_leaves = sorted(estate_leaves - set(category_counts))

assert len(needs) == len(results) == state['total'] == state['completed'] == 10_000
assert state['useLlmCompose'] is False
assert all(source.startswith('template+') for source in source_counts)
assert all(row.get('aiInvoked') is False for row in results)
assert matched_expected_category == len(needs)

profile = {
    'rows': len(needs),
    'source_counts': dict(sorted(source_counts.items())),
    'tone_counts': dict(sorted(tone_counts.items())),
    'category_counts': dict(sorted(category_counts.items())),
    'expected_category_matches': matched_expected_category,
    'parser_passed': sum(row.get('ok') is True for row in results),
    'parser_failed': sum(row.get('ok') is not True for row in results),
    'ai_invoked': sum(row.get('aiInvoked') is True for row in results),
    'rows_with_resolved_city': sum(bool(row.get('resolvedCity')) for row in results),
    'distinct_resolved_cities': len(resolved_cities),
    'rows_with_resolved_neighborhood': sum(bool(row.get('resolvedNeighborhood')) for row in results),
    'distinct_resolved_neighborhoods': len(resolved_neighborhoods),
    'duplicate_normalized_text_rows': duplicate_text_rows,
    'real_estate_leaf_coverage': f'{len(estate_categories)}/{len(estate_leaves)}',
    'present_real_estate_leaves': sorted(estate_categories),
    'missing_real_estate_leaves': missing_estate_leaves,
    'need_row_fields': sorted(set().union(*(row.keys() for row in needs))),
    'has_consent_provenance_or_human_review_fields': any(
        key in row for row in needs for key in ('consent', 'provenance', 'reviewed', 'humanReviewed')
    ),
    'has_laya_typed_question_targets': any(
        key in row for row in needs for key in ('questions', 'typedDecisions', 'gold')
    ),
}
profile

### Observed result snapshot

| Check | Result |
|---|---:|
| Rows | 10,000 |
| Unique normalized texts | 22 |
| Duplicate rows | 9,978 (99.78%) |
| Parser fixture passed | 10,000 / 10,000 |
| AI invoked | 0 |
| Distinct resolved cities / neighborhoods | 9 / 16 |
| Real-estate leaves represented | 6 / 18 |
| Laya typed-decision targets / consent provenance | absent / absent |

Present property leaves: `apartment-rent`, `land-sale`, `office-rent`, `shop-rent`, `suite-apartment-rent`, `villa-sale`. This is a snapshot of the local run; the code above recomputes it from the source files.

### Interpretation

A 100% pass rate is not a model-accuracy estimate: the generated text and expected category/location were derived from the same profile, and no independent seeker or reviewer supplied ground truth. The 9,978 duplicate rows reduce the effective text variety to 22 normalized inputs. The result schema does not provide Laya question definitions or typed-decision targets, nor consent or per-row provenance. City/neighborhood resolution therefore measures that the fixture profile was replayed, not general Persian location extraction.

## Takeaways

- Keep these rows as parser stress-test fixtures; exclude them from real-need fine-tuning and held-out evaluation.
- Do not count their apparent city/neighborhood coverage as evidence of production location accuracy.
- The genuine first-party path currently does not capture published needs because the publish caller supplies no required consent evidence. Fixing that path requires a voluntary, versioned opt-in and privacy-safe audit record before collection begins.
- Divar transformations may be considered only as a separately labeled synthetic auxiliary set; an offer does not prove a person's actual demand or preferences.